In [ ]:
import os
import numpy as np
import keras
from keras import layers
from tensorflow import data as tf_data
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
num = 0
for folder_name in ("elipse", "spiral", "irregular", "lens"):
    folder_path = os.path.join("canny", folder_name)
    print(folder_path)
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        num += 1
        

print(f"Found {num} images.")

In [ ]:
image_size = (424, 424)
batch_size = 32
# path = "C:\\Users\\gosia\\Dokumenty\\Nauka\\VII sem\\Inżynierka\\canny"
path = ".\\canny"

if os.path.exists(path):
    # Proceed with data loading
    print("Path found")
else:
    print("File not found.")

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    directory=path,
    label_mode="categorical",
    validation_split=0.3,
    subset="both",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size
)

In [ ]:
val_batches = tf.data.experimental.cardinality(val_ds)
test_ds = val_ds.take((2*val_batches) // 3)
val_ds = val_ds.skip((2*val_batches) // 3)

In [ ]:
# plt.figure(figsize=(10, 10))
# for images, labels in train_ds.take(1):
#     for i in range(9):
#         ax = plt.subplot(3, 3, i + 1)
#         plt.imshow(np.array(images[i]).astype("uint8"))
#         plt.title(labels[i])
#         plt.axis("off")

In [ ]:
data_augmentation_layers = [
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
]


def data_augmentation(images):
    for layer in data_augmentation_layers:
        images = layer(images)
    return images

augmented_train_ds = train_ds.map(lambda x, y: (data_augmentation(x), y))

In [ ]:
train_ds = train_ds.map(
    lambda img, label: (data_augmentation(img), label),
    num_parallel_calls=tf_data.AUTOTUNE,
)
# Prefetching samples in GPU memory helps maximize GPU utilization.
train_ds = train_ds.prefetch(tf_data.AUTOTUNE)
val_ds = val_ds.prefetch(tf_data.AUTOTUNE)

In [ ]:
def make_model(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Entry block
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.Conv2D(32, 3, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    # x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    for size in [64, 128]:
        x = layers.Activation("relu")(x)
        x = layers.Conv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.Activation("relu")(x)
        x = layers.Conv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    if num_classes == 2:
        units = 1
    else:
        units = num_classes
    x = layers.Dropout(0.25)(x)
    # We specify activation=None so as to return logits
    
    outputs = layers.Dense(units, activation="sigmoid")(x)
    return keras.Model(inputs, outputs)

def make_model2(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Entry block
    x = layers.Rescaling(1.0 / 255)(inputs)
    x = layers.Conv2D(32, 11, strides=2, padding="same")(x)
    x = layers.MaxPooling2D(7, strides=2, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D(2, strides=2, padding="same")(x)

    x = layers.Activation("relu")(x)
    x = layers.Conv2D(128, 5, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, strides=2, padding="same")(x)

    x = layers.Activation("relu")(x)
    x = layers.Conv2D(256, 5, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, strides=2, padding="same")(x)

    x = layers.Activation("relu")(x)
    x = layers.Conv2D(256, 5, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, strides=2, padding="same")(x)

    x = layers.Activation("relu")(x)
    x = layers.Conv2D(512, 5, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2, strides=2, padding="same")(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    if num_classes == 2:
        units = 1
    else:
        units = num_classes
    x = layers.Dropout(0.25)(x)
    # We specify activation=None so as to return logits
    
    outputs = layers.Dense(units, activation="sigmoid")(x)
    return keras.Model(inputs, outputs)

model = make_model2(input_shape=image_size + (3,), num_classes=4)
# keras.utils.plot_model(model, show_shapes=True)

In [ ]:
# def make_model(input_shape, num_classes):
#     inputs = keras.Input(shape=input_shape)

#     # Entry block
#     x = layers.Rescaling(1.0 / 255)(inputs)
#     x = layers.Conv2D(32, 3, strides=2, padding="same")(x)
#     x = layers.BatchNormalization()(x)
#     x = layers.Activation("relu")(x)

#     previous_block_activation = x  # Set aside residual


#     for size in [64, 128]:
        
#         x = layers.Conv2D(size, 3, padding="same")(x)
#         x = layers.BatchNormalization()(x)
#         x = layers.Activation("relu")(x)
#         # x = layers.Activation("relu")(x)
#         # x = layers.Conv2D(size, 3, padding="same")(x)
#         # x = layers.BatchNormalization()(x)

#         x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

#         # Project residual
#         residual = layers.Conv2D(size, 1, strides=2, padding="same")(
#             previous_block_activation
#         )
#         x = layers.add([x, residual])  # Add back residual
#         previous_block_activation = x  # Set aside next residual

#     # x = layers.SeparableConv2D(1024, 3, padding="same")(x)
#     # x = layers.BatchNormalization()(x)
#     # x = layers.Activation("relu")(x)

#     x = layers.GlobalAveragePooling2D()(x)
#     if num_classes == 2:
#         units = 1
#     else:
#         units = num_classes
#     # x = layers.Dense(128, activation="relu")(x)
#     x = layers.Dropout(0.25)(x)
#     # We specify activation=None so as to return logits
#     outputs = layers.Dense(units, activation=None)(x)
#     return keras.Model(inputs, outputs)


# model = make_model(input_shape=image_size + (3,), num_classes=4)
# # keras.utils.plot_model(model, show_shapes=True)

In [ ]:
epochs = 100

callbacks = [
    keras.callbacks.ModelCheckpoint("save_at_{epoch}.keras"),
]
model.compile(
    optimizer=keras.optimizers.Adam(3e-4),
    loss='CategoricalCrossentropy',
    metrics=[keras.metrics.BinaryAccuracy(name="acc")],
)
hist = model.fit(
    train_ds,
    epochs=epochs,
    callbacks=callbacks,
    validation_data=val_ds,
)

In [ ]:
h = hist.history
e = range(1, epochs+1)

# Define a figure, in this case subplots with 1 row with 3 columns
fig, (ax1, ax2) = plt.subplots(1,2 , 
                                    figsize = (16,4))
# Plot 1, Loss
ax1.plot(e, h['loss'], 'g.', label='Training loss')    
ax1.plot(e, h['val_loss'], 'g', label='Validation loss')    
ax1.set_title('Training and validation loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend() 

# Plot 2, Accuracy
ax2.plot(e, h['acc'], 'r.', label='Training acc')    
ax2.plot(e, h['val_acc'], 'r', label='Validation acc')    
ax2.set_title('Training and validation accuracy')    
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Accuracy')
ax2.legend() 

fig.show()


In [ ]:
results = model.evaluate(test_ds)